# dist-send-recv-pair — worked example 2: Gather tensors from all ranks to rank 0 using send/recv

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dist-send-recv-pair`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Gathering is the reverse of scattering: each non-zero rank sends its local tensor to rank 0, and rank 0 posts one `dist.recv` per sender. The sender side is simple — each rank calls `dist.send` once. Rank 0 collects into a list of receive buffers, one per source rank, then all gathered tensors are available at rank 0.

## Worked solution

With `world_size=4`, each rank holding a scalar tensor `[rank_value]`:

**Rank 1, 2, 3 (senders):**
- `dist.send(torch.tensor([my_value]), dst=0)`.

**Rank 0 (gatherer):**
- Loops `src = 1, 2, 3`.
- For each: allocates `buf = torch.zeros(1)`, calls `dist.recv(buf, src=src)`, appends `buf` to `gathered`.
- Also prepends its own value: `gathered[0] = my_tensor`.

**Result at rank 0:** `gathered[i]` contains the value that rank `i` held. Ranks 1–3 do not hold the gathered result.

In [ ]:
import os
import torch
import torch.distributed as dist
import datetime
from queue import Queue

def gather_worker(rank, world_size, port, my_value, out_queue):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    dist.init_process_group(
        backend='gloo', rank=rank, world_size=world_size,
        timeout=datetime.timedelta(seconds=20)
    )
    my_tensor = torch.tensor([float(my_value)], dtype=torch.float32)
    if rank != 0:
        # Non-root: send to rank 0 and done
        dist.send(my_tensor, dst=0)
        out_queue.put((rank, None))  # non-root has no gathered result
    else:
        # Root: collect from every other rank
        gathered = [None] * world_size
        gathered[0] = my_tensor.clone()
        for src in range(1, world_size):
            buf = torch.zeros(1, dtype=torch.float32)
            dist.recv(buf, src=src)
            gathered[src] = buf.clone()
        result = [t.item() for t in gathered]
        out_queue.put((rank, result))
    dist.destroy_process_group()

# Simulate with a printout (real test uses mp.spawn)
print('Gather simulation (world_size=4):')
values = [10.0, 20.0, 30.0, 40.0]
print('Each rank sends its value to rank 0.')
print('Rank 0 gathers:', values)
print('Non-root ranks: no gathered result.')